# 📊 Query Generation Analysis
Analisi delle query generate in **Pandas** e **SQL**: success/failure rate, distribuzione per difficoltà, tabelle coinvolte e keyword più utilizzate.

---
## 0. Setup & Caricamento Dati

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import numpy as np
from collections import Counter

# ── Stile globale ──────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 130,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

PALETTE = {
    'success': '#2ECC71',
    'failure': '#E74C3C',
    'PANDAS':  '#3498DB',
    'SQL':     '#9B59B6',
    'simple':  '#1ABC9C',
    'hard':    '#F39C12',
    'challenging': '#E74C3C',
}

DIFF_ORDER = ['simple', 'hard', 'challenging']

# ── Caricamento ────────────────────────────────────────────────────────────────
with open(r'D:\orqa\socrata\nyc\candidates_discovery\generated_queries.json', 'r') as f:
    raw = json.load(f)

print('Lingue trovate:', list(raw.keys()))
for lang, entries in raw.items():
    print(f'  {lang}: {len(entries)} entry')

In [ ]:
# ── Flattenizzazione in un DataFrame ──────────────────────────────────────────
records_entry = []   # una riga per entry (per success/failure)
records_query = []   # una riga per query  (per keyword/difficoltà)

for lang, entries in raw.items():
    for entry_id, entry in entries.items():
        n_tables = len(entry.get('tables', {}))
        status   = entry.get('status', 'unknown')
        queries  = entry.get('data', {}).get('queries', [])

        records_entry.append({
            'language': lang,
            'entry_id': entry_id,
            'status':   status,
            'n_tables': n_tables,
            'n_queries': len(queries),
        })

        for q in queries:
            records_query.append({
                'language':   lang,
                'entry_id':   entry_id,
                'status':     status,
                'difficulty': q.get('difficulty', 'unknown'),
                'n_tables':   n_tables,
                'keywords':   q.get('keywords', {}),
            })

df_entry = pd.DataFrame(records_entry)
df_query = pd.DataFrame(records_query)
df_query['difficulty'] = pd.Categorical(df_query['difficulty'], categories=DIFF_ORDER, ordered=True)

print(f'Entry totali: {len(df_entry)}')
print(f'Query totali: {len(df_query)}')
df_entry.head()

---
## 1. Tasso di Success vs Failure
### 1a. Overview Generale

In [ ]:
overall = df_entry['status'].value_counts()
total   = overall.sum()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Success vs Failure – Panoramica Generale', fontsize=14, fontweight='bold')

# ── Pie ───────────────────────────────────────────────────────────────────────
colors = [PALETTE.get(s, '#95A5A6') for s in overall.index]
wedges, texts, autotexts = axes[0].pie(
    overall.values, labels=overall.index,
    colors=colors, autopct='%1.1f%%',
    startangle=90, pctdistance=0.75,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
for at in autotexts:
    at.set_fontsize(12); at.set_fontweight('bold')
axes[0].set_title('Distribuzione %')

# ── Bar con conteggi ──────────────────────────────────────────────────────────
bars = axes[1].bar(overall.index, overall.values, color=colors, edgecolor='white', width=0.5)
for bar, val in zip(bars, overall.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val}\n({val/total*100:.1f}%)', ha='center', va='bottom', fontsize=11)
axes[1].set_ylabel('Numero di entry')
axes[1].set_title('Conteggio Assoluto')
axes[1].set_ylim(0, overall.max() * 1.2)

plt.tight_layout()
plt.show()
print(f'\nTotale entry: {total} | Success: {overall.get("success",0)} | Failure: {overall.get("failure",0)}')

### 1b. Success/Failure per Linguaggio

In [ ]:
by_lang = df_entry.groupby(['language', 'status']).size().unstack(fill_value=0)
by_lang_pct = by_lang.div(by_lang.sum(axis=1), axis=0) * 100

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Success vs Failure per Linguaggio', fontsize=14, fontweight='bold')

# Conteggi assoluti
bar_colors = [PALETTE.get(s, '#95A5A6') for s in by_lang.columns]
by_lang.plot(kind='bar', ax=axes[0], color=bar_colors, edgecolor='white', rot=0)
axes[0].set_title('Conteggio Assoluto')
axes[0].set_ylabel('Entry')
axes[0].legend(title='Status')
for container in axes[0].containers:
    axes[0].bar_label(container, fmt='%d', padding=3)

# Percentuali stacked
bar_colors_pct = [PALETTE.get(s, '#95A5A6') for s in by_lang_pct.columns]
by_lang_pct.plot(kind='bar', stacked=True, ax=axes[1], color=bar_colors_pct, edgecolor='white', rot=0)
axes[1].set_title('Distribuzione %')
axes[1].set_ylabel('%')
axes[1].set_ylim(0, 115)
axes[1].legend(title='Status', loc='upper right')
for container in axes[1].containers:
    axes[1].bar_label(container, fmt='%.1f%%', label_type='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()
print(by_lang_pct.round(1))

### 1c. Success/Failure per Difficoltà

In [ ]:
# Difficoltà derivata dalla prima query disponibile di ogni entry
first_q = df_query.groupby(['language','entry_id'])['difficulty'].first().reset_index()
df_entry_d = df_entry.merge(first_q, on=['language','entry_id'], how='left')

by_diff = (df_entry_d.groupby(['difficulty','status'])
           .size().unstack(fill_value=0)
           .reindex(DIFF_ORDER))
by_diff_pct = by_diff.div(by_diff.sum(axis=1), axis=0) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Success vs Failure per Difficoltà', fontsize=14, fontweight='bold')

bar_colors = [PALETTE.get(s, '#95A5A6') for s in by_diff.columns]
by_diff.plot(kind='bar', ax=axes[0], color=bar_colors, edgecolor='white', rot=0)
axes[0].set_title('Conteggio Assoluto')
axes[0].set_ylabel('Entry')
axes[0].legend(title='Status')
for container in axes[0].containers:
    axes[0].bar_label(container, fmt='%d', padding=3)

by_diff_pct.plot(kind='bar', stacked=True, ax=axes[1], color=bar_colors, edgecolor='white', rot=0)
axes[1].set_title('Distribuzione %')
axes[1].set_ylabel('%')
axes[1].set_ylim(0, 115)
axes[1].legend(title='Status', loc='upper right')
for container in axes[1].containers:
    axes[1].bar_label(container, fmt='%.1f%%', label_type='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()
print(by_diff_pct.round(1))

---
## 2. Query Generate con Successo per Difficoltà

In [ ]:
# Solo le entry con status success
df_ok = df_query[df_query['status'] == 'success'].copy()

# ── Conteggio query per difficoltà e linguaggio ───────────────────────────────
pivot = (df_ok.groupby(['difficulty','language'])
         .size().unstack(fill_value=0)
         .reindex(DIFF_ORDER))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Query Genrate con Successo per Difficoltà', fontsize=14, fontweight='bold')

# Per linguaggio
lang_colors = [PALETTE.get(l) for l in pivot.columns]
pivot.plot(kind='bar', ax=axes[0], color=lang_colors, edgecolor='white', rot=0)
axes[0].set_title('Per Linguaggio')
axes[0].set_ylabel('Numero di query')
axes[0].legend(title='Linguaggio')
for container in axes[0].containers:
    axes[0].bar_label(container, fmt='%d', padding=3)

# Totali per difficoltà
totals = pivot.sum(axis=1)
diff_colors = [PALETTE.get(d) for d in DIFF_ORDER]
bars = axes[1].bar(DIFF_ORDER, totals.values, color=diff_colors, edgecolor='white', width=0.5)
for bar, val in zip(bars, totals.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 str(int(val)), ha='center', va='bottom', fontsize=12, fontweight='bold')
axes[1].set_title('Totale per Difficoltà')
axes[1].set_ylabel('Numero di query')
axes[1].set_ylim(0, totals.max() * 1.15)

plt.tight_layout()
plt.show()
print(pivot)

---
## 3. Numero di Tabelle Coinvolte per Difficoltà

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('Tabelle Coinvolte per Difficoltà', fontsize=14, fontweight='bold')

# ── Heatmap: difficoltà × n_tabelle ──────────────────────────────────────────
heat = (df_ok.groupby(['difficulty','n_tables'])
        .size().unstack(fill_value=0)
        .reindex(DIFF_ORDER))

im = axes[0].imshow(heat.values, cmap='YlOrRd', aspect='auto')
axes[0].set_xticks(range(len(heat.columns)))
axes[0].set_xticklabels(heat.columns)
axes[0].set_yticks(range(len(heat.index)))
axes[0].set_yticklabels(heat.index)
axes[0].set_xlabel('N° Tabelle')
axes[0].set_title('Heatmap (query count)')
plt.colorbar(im, ax=axes[0], shrink=0.8)
for i in range(len(heat.index)):
    for j in range(len(heat.columns)):
        v = heat.values[i, j]
        if v > 0:
            axes[0].text(j, i, str(v), ha='center', va='center',
                         fontsize=9, color='black' if v < heat.values.max()*0.6 else 'white')

# ── Boxplot n_tabelle per difficoltà ─────────────────────────────────────────
bp_data = [df_ok[df_ok['difficulty']==d]['n_tables'].values for d in DIFF_ORDER]
bp = axes[1].boxplot(bp_data, labels=DIFF_ORDER, patch_artist=True, notch=False,
                     medianprops={'color':'black','linewidth':2})
for patch, d in zip(bp['boxes'], DIFF_ORDER):
    patch.set_facecolor(PALETTE.get(d, '#95A5A6'))
    patch.set_alpha(0.8)
axes[1].set_ylabel('N° Tabelle')
axes[1].set_title('Distribuzione (Boxplot)')

# ── Media tabelle per difficoltà e linguaggio ─────────────────────────────────
avg = (df_ok.groupby(['difficulty','language'])['n_tables']
       .mean().unstack().reindex(DIFF_ORDER))
x = np.arange(len(DIFF_ORDER))
w = 0.35
for i, lang in enumerate(avg.columns):
    axes[2].bar(x + i*w, avg[lang].values, width=w,
                label=lang, color=PALETTE.get(lang), edgecolor='white', alpha=0.9)
axes[2].set_xticks(x + w/2)
axes[2].set_xticklabels(DIFF_ORDER)
axes[2].set_ylabel('Media tabelle')
axes[2].set_title('Media Tabelle per Linguaggio')
axes[2].legend(title='Linguaggio')
for container in axes[2].containers:
    axes[2].bar_label(container, fmt='%.1f', padding=3, fontsize=9)

plt.tight_layout()
plt.show()
print('\nMedia tabelle per difficoltà:')
print(df_ok.groupby('difficulty')['n_tables'].agg(['mean','median','min','max']).round(2))

---
## 4. Keyword Analysis
### 4a. Definizione Blacklist

In [ ]:
# Blacklist: keyword troppo generiche o condivise tra entrambi i linguaggi
BLACKLIST = {
    # strutturali SQL
    'SELECT', 'FROM', 'WHERE', 'ON', 'AS', 'AND', 'OR', 'NOT',
    'IN', 'IS', 'NULL', 'BY', 'CASE', 'WHEN', 'THEN', 'ELSE', 'END',
    # pandas base
    'values', 'index', 'dtype', 'dtypes', 'columns', 'shape', 'len',
    # numeri / simboli / punteggiatura che compaiono come chiave
    '', ' ',
}

def aggregate_keywords(df_subset, blacklist=BLACKLIST):
    """Aggrega tutti i keywords di una subset di df_query."""
    counter = Counter()
    for kw_dict in df_subset['keywords']:
        for kw, cnt in kw_dict.items():
            kw_clean = kw.strip()
            if kw_clean.upper() not in {b.upper() for b in blacklist} and kw_clean:
                counter[kw_clean] += cnt
    return counter

kw_pandas = aggregate_keywords(df_ok[df_ok['language']=='PANDAS'])
kw_sql    = aggregate_keywords(df_ok[df_ok['language']=='SQL'])

print(f'Keyword univoche PANDAS (post-blacklist): {len(kw_pandas)}')
print(f'Keyword univoche SQL    (post-blacklist): {len(kw_sql)}')
print('\nTop 10 PANDAS:', kw_pandas.most_common(10))
print('Top 10 SQL:   ', kw_sql.most_common(10))

### 4b. Top Keywords – PANDAS vs SQL

In [ ]:
TOP_N = 20

def kw_to_df(counter, top_n=TOP_N):
    items = counter.most_common(top_n)
    return pd.DataFrame(items, columns=['keyword','count'])

df_kp = kw_to_df(kw_pandas)
df_ks = kw_to_df(kw_sql)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
fig.suptitle(f'Top {TOP_N} Keyword per Linguaggio (post-blacklist)', fontsize=14, fontweight='bold')

for ax, df_k, lang in zip(axes, [df_kp, df_ks], ['PANDAS', 'SQL']):
    df_sorted = df_k.sort_values('count')
    bars = ax.barh(df_sorted['keyword'], df_sorted['count'],
                   color=PALETTE[lang], edgecolor='white', alpha=0.88)
    ax.bar_label(bars, fmt='%d', padding=4, fontsize=9)
    ax.set_title(lang, fontsize=13, fontweight='bold', color=PALETTE[lang])
    ax.set_xlabel('Occorrenze totali')
    ax.set_xlim(0, df_sorted['count'].max() * 1.18)

plt.tight_layout()
plt.show()

### 4c. Keyword per Difficoltà (PANDAS vs SQL)

In [ ]:
TOP_PER_DIFF = 8

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle(f'Top {TOP_PER_DIFF} Keyword per Linguaggio × Difficoltà', fontsize=14, fontweight='bold')

for row, lang in enumerate(['PANDAS', 'SQL']):
    for col, diff in enumerate(DIFF_ORDER):
        ax = axes[row][col]
        subset = df_ok[(df_ok['language']==lang) & (df_ok['difficulty']==diff)]
        counter = aggregate_keywords(subset)
        top = counter.most_common(TOP_PER_DIFF)
        if not top:
            ax.text(0.5, 0.5, 'Nessuna keyword', ha='center', va='center', transform=ax.transAxes)
            ax.set_title(f'{lang} – {diff}')
            continue
        kws, cnts = zip(*top)
        # Colori sfumati per importanza
        base = PALETTE[lang]
        alphas = np.linspace(0.95, 0.45, len(kws))
        bars = ax.barh(kws[::-1], cnts[::-1],
                       color=[base]*len(kws), alpha=1.0, edgecolor='white')
        for bar, alpha in zip(bars, alphas[::-1]):
            bar.set_alpha(alpha)
        ax.bar_label(bars, fmt='%d', padding=3, fontsize=8)
        ax.set_title(f'{lang} – {diff}', fontweight='bold',
                     color=PALETTE[lang] if col==0 else 'black')
        ax.set_xlabel('Occorrenze')
        ax.set_xlim(0, max(cnts)*1.22)

plt.tight_layout()
plt.show()

### 4d. Keyword Esclusive vs Condivise

In [ ]:
all_pandas_kw = set(k.lower() for k in kw_pandas.keys())
all_sql_kw    = set(k.lower() for k in kw_sql.keys())

only_pandas  = all_pandas_kw - all_sql_kw
only_sql     = all_sql_kw - all_pandas_kw
shared       = all_pandas_kw & all_sql_kw

print(f'Keyword SOLO PANDAS  ({len(only_pandas)}): {sorted(only_pandas)[:20]}')
print(f'Keyword SOLO SQL     ({len(only_sql)}):    {sorted(only_sql)[:20]}')
print(f'Keyword CONDIVISE    ({len(shared)}):       {sorted(shared)}')

# ── Venn-like bar chart ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
categories = ['Solo PANDAS', 'Condivise', 'Solo SQL']
counts     = [len(only_pandas), len(shared), len(only_sql)]
colors     = [PALETTE['PANDAS'], '#7F8C8D', PALETTE['SQL']]
bars = ax.bar(categories, counts, color=colors, edgecolor='white', width=0.5)
ax.bar_label(bars, fmt='%d', padding=5, fontsize=13, fontweight='bold')
ax.set_ylabel('N° keyword univoche')
ax.set_title('Keyword Esclusive vs Condivise', fontsize=13, fontweight='bold')
ax.set_ylim(0, max(counts)*1.2)
plt.tight_layout()
plt.show()

---
## 5. Riepilogo Statistico

In [ ]:
print('=' * 60)
print('RIEPILOGO STATISTICO')
print('=' * 60)

for lang in ['PANDAS', 'SQL']:
    sub = df_entry[df_entry['language']==lang]
    n_ok  = (sub['status']=='success').sum()
    n_fail = (sub['status']=='failure').sum()
    print(f'\n{lang}')
    print(f'  Entry totali : {len(sub)}')
    print(f'  Success      : {n_ok} ({n_ok/len(sub)*100:.1f}%)')
    print(f'  Failure      : {n_fail} ({n_fail/len(sub)*100:.1f}%)')
    qsub = df_ok[df_ok['language']==lang]
    print(f'  Query ok     : {len(qsub)}')
    print(f'  Media tabelle: {qsub["n_tables"].mean():.2f}')
    top3 = aggregate_keywords(qsub).most_common(3)
    print(f'  Top 3 kw     : {top3}')

print()
print('Per difficoltà (query ok):')
print(df_ok.groupby(['language','difficulty']).size().unstack(fill_value=0))